In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent

import os
from dotenv import load_dotenv
load_dotenv()

/tmp/ipykernel_3602/3500639950.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


True

In [14]:
# db = SQLDatabase.from_uri("sqlite:///data/employees_db-full-1.0.6.db")
db = SQLDatabase.from_uri("sqlite:///../data/employees_db-full-1.0.6.db")

try:
    tables = db.get_usable_table_names()
    print("Successfully connected to the database.")
    print(f"Number of tables found: {len(tables)}, Tables: {', '.join(tables)}")
except Exception as e:
    print("Failed to connect to the database.")
    print(f"Error: {e}")

schema = db.get_table_info()


Successfully connected to the database.
Number of tables found: 6, Tables: departments, dept_emp, dept_manager, employees, salaries, titles


In [15]:
llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        project="ai-framework-projects",
        temperature=0
        )

In [18]:
@tool
def get_database_schema(table_name: str = None):
    """
    Get the database schema for generating the SQL queries.
    Use this first to understand the table structure and relationships before generating SQL queries.
    """
    if table_name:
        try:
            tables = db.get_usable_table_names()
            if table_name.lower() in [t.lower() for t in tables]:
                result = db.get_table_info([table_name])
                print("Retrived the schema for the table: ", table_name)
                return result
            else:
                return f"Table '{table_name}' does not exist in the database. Available tables: {', '.join(tables)}"
        except Exception as e:
            return f"Error getting the schema for the table {table_name}: {e}"
    else:
        print("Retrived Full database schema.")
        return db.get_table_info()

In [19]:
get_database_schema.invoke({'table_name': 'dept_emp'})

Retrived the schema for the table:  dept_emp


'\nCREATE TABLE dept_emp (\n\temp_no INTEGER NOT NULL, \n\tdept_no CHAR(4) NOT NULL, \n\tfrom_date DATE NOT NULL, \n\tto_date DATE NOT NULL, \n\tPRIMARY KEY (emp_no, dept_no), \n\tFOREIGN KEY(dept_no) REFERENCES departments (dept_no), \n\tFOREIGN KEY(emp_no) REFERENCES employees (emp_no)\n)\n\n/*\n3 rows from dept_emp table:\nemp_no\tdept_no\tfrom_date\tto_date\n10001\td005\t1986-06-26\t9999-01-01\n10002\td007\t1996-08-03\t9999-01-01\n10003\td004\t1995-12-03\t9999-01-01\n*/'